# Tiny heads on tree bits — MagicTelescope

**What this run asks.** In the last run, even the smallest MLP and TabResNet (width 32) pushed the training score to 1, and no regularisation stopped that. So this run makes the heads much smaller. **The old smallest size becomes the new "large"**, and the width is halved four more times, down to a head only about twice the size of logistic regression. **The transformer is not in this run.**

### The sizes
Only the width changes. Every MLP keeps 2 hidden layers and every TabResNet keeps 2 residual blocks.

| size | width | MLP weights | TabResNet weights | note |
|---|---|---|---|---|
| **w32 large** | 32 | 47,810 | 55,202 | same as last run's "small" |
| **w16 medium** | 16 | 23,650 | 25,554 | |
| **w8 small** | 8 | 11,762 | 12,266 | |
| **w4 tiny** | 4 | 5,866 | 6,006 | |
| **w2 micro** | 2 | 2,930 | 2,972 | about 2× logistic regression |
| logistic regression | – | 1,455 | | the reference |

### Everything else stays exactly as in the last run
Same forest (100 trees, at most 6 levels deep), same trimmed answers (top 4 levels, 1,454 bits), same honest out-of-bag rule for training rows, same split, and the same six regularisation settings: none, hidden dropout 0.3, weight decay 0.3, input dropout 0.3, input dropout 0.5, all three. Training: AdamW, learning rate 0.001, batch 256, up to 150 epochs, stop after 30 epochs without improvement on validation, keep the best validation epoch. The test set is never used to choose anything.

### What to look for
* At which size does the **training score stop reaching 1**?
* Does the **overfitting gap** shrink as the head gets smaller?
* Does the **test score** hold, rise or fall? Where does it meet logistic regression (0.9038)?
* **w32** repeats last run's "small" runs, so it also checks that the results reproduce (cell 8 compares them).

| part | what runs | runs | rough time |
|---|---|---|---|
| **A** | logistic regression reference, on both encodings | 2 | about 5 min |
| **B** | MLP and TabResNet × 5 widths × 6 regularisation settings | 60 | about 30–45 min |

### Before you start
**Runtime → Change runtime type → GPU** (any GPU is fine), then **Runtime → Run all**. Cell 2 asks for Drive access once.

### If the runtime disconnects
Run all cells again. Each finished run is copied to Drive the moment it ends and is skipped next time. A run that was in progress continues from its last checkpoint (saved every 10 epochs). The forest and the bits are cached on Drive, so they are never rebuilt.

### At the end
The last cell downloads **`deep_head_tiny_<date>.zip`**: every run's log and results, the tables (CSV and Excel), the charts and a README. A copy also stays on Drive in `MyDrive/TKCE/deep_head_tiny/`.

In [ ]:
# 1 · check the GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NO GPU — set Runtime > Change runtime type > GPU, then run again')

In [ ]:
# 2 · mount Drive and lay out the folders that make a disconnect harmless
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, shutil
DRIVE = '/content/drive/MyDrive/TKCE/deep_head_tiny'
# The forest and the bits are shared with the earlier deep-head run. Every cache
# file name carries the settings it was built with, so sharing is safe.
CACHE = '/content/drive/MyDrive/TKCE/deep_head/cache'
CKPT  = f'{DRIVE}/ckpt'     # mid-training checkpoints, every 10 epochs
RUNS  = f'{DRIVE}/runs'     # each finished run, copied over the moment it ends
for d in (CACHE, CKPT, RUNS): os.makedirs(d, exist_ok=True)
print('results go to  ', DRIVE)
print('shared cache at', CACHE, f'({len(os.listdir(CACHE))} files already there)')

In [ ]:
# 3 · get the code
%cd /content
!git clone -q https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull -q
!git log -1 --format='code version: %h  %s'
!pip install -q openml lightgbm xgboost 2>&1 | grep -v 'already satisfied' | tail -2

In [ ]:
# 4 · restore anything an earlier session already finished
ROOT = 'results/deep_head_tiny'
os.makedirs(ROOT, exist_ok=True)
if glob.glob(f'{RUNS}/*/*/deephead_*.json'):
    shutil.copytree(RUNS, ROOT, dirs_exist_ok=True)
    done = glob.glob(f'{ROOT}/*/*/deephead_*.json')
    print(f'restored {len(done)} finished runs from Drive; they will be skipped')
else:
    print('no earlier runs on Drive; starting fresh')
ck = glob.glob(f'{CKPT}/ck_*.pt')
print(f'{len(ck)} run(s) stopped mid-training; they will continue from their checkpoint'
      if ck else 'no runs waiting to be continued')
SUITE = f'--grid tiny --root {ROOT} --cache-dir {CACHE} --ckpt-dir {CKPT} --sync-dir {RUNS}'
print('\n' + SUITE)

## Part A — the logistic regression reference
Logistic regression from scikit-learn on the same answers, with its strength C tried at 9 values and the best kept on validation. Two runs (top 4 levels, and all levels). The first session also builds the forest and the tree baselines here, unless they are already cached.

In [ ]:
!python -u run_deep_head_suite.py {SUITE} --part A 2>&1 | tee -a {ROOT}/suite.log

## Part B — the width ladder
60 runs: MLP and TabResNet, five widths each, six regularisation settings each. The console shows one line every 15 epochs per run; every run's full log is saved next to its results. Safe to interrupt and run again.

In [ ]:
!python -u run_deep_head_suite.py {SUITE} --part B 2>&1 | tee -a {ROOT}/suite.log

In [ ]:
# 5 · the summary table: best run per head, chosen on validation, never on test
!python -u run_deep_head_suite.py {SUITE} --aggregate-only 2>&1 | grep -v '^\[suite\] figures'

In [ ]:
# 6 · the capacity ladder: what happens as the head gets smaller?
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
t = pd.read_csv(f'{ROOT}/suite_long.csv')
lin = t[(t['head'] == 'linear') & (t['bits'] == 'shallow')]
LIN = float(lin['test'].iloc[0]) if len(lin) else float('nan')
CEIL = float(t['tree_ceiling'].iloc[0])
sh = t[(t['bits'] == 'shallow') & (t['head'] != 'linear')]
NAME = {'w32': 'w32 large', 'w16': 'w16 medium', 'w8': 'w8 small', 'w4': 'w4 tiny', 'w2': 'w2 micro'}
print(f'logistic regression {LIN:.4f}  |  best tree {CEIL:.4f}  |  differences under 0.01 are ties')
if len(lin):
    print(f'logistic regression train score {float(lin["train"].iloc[0]):.4f}, gap {float(lin["gap"].iloc[0]):.4f}')
print('Each row summarises the six regularisation settings at one size. Train is measured at the epoch kept.\n')
rows = []
for h in ['mlp', 'tabresnet']:
    for s in ['w32', 'w16', 'w8', 'w4', 'w2']:
        g = sh[(sh['head'] == h) & (sh['size'] == s)]
        if not len(g):
            continue
        pick = g.sort_values('val').iloc[-1]          # chosen on validation
        rows.append({'head': h, 'size': NAME[s], 'weights': int(g['n_params'].iloc[0]), 'runs': len(g),
                     'train (median)': g['train'].median(), 'train acc (median)': g['train_acc'].median(),
                     'runs with train >= 0.99': int((g['train'] >= 0.99).sum()),
                     'gap (median)': g['gap'].median(), 'test (median)': g['test'].median(),
                     'test log loss (median)': g['test_logloss'].median(),
                     'picked on val': pick['reg'], 'its test': pick['test'],
                     'its test vs logistic': pick['test'] - LIN})
print(pd.DataFrame(rows).round(4).to_string(index=False))

# runs that used all 150 epochs may still have been improving: say so plainly
late = sh[(sh['epochs_run'] >= 150) | (sh['best_epoch'] > 120)]
print('\nruns that may have needed more than 150 epochs:',
      'none' if late.empty else '')
if len(late):
    print(late[['head', 'size', 'reg', 'best_epoch', 'epochs_run', 'val', 'test']].to_string(index=False))

In [ ]:
# 7 · every run, one table per head: test score and gap by size and regularisation
for h in ['mlp', 'tabresnet']:
    g = sh[sh['head'] == h]
    if not len(g):
        continue
    order = ['w32', 'w16', 'w8', 'w4', 'w2']
    regs = ['none', 'hidden', 'decay', 'input', 'input5', 'all']
    for col in ['train', 'test', 'gap']:
        p = g.pivot_table(index='size', columns='reg', values=col).reindex(index=order, columns=regs)
        p.index = [NAME[i] for i in p.index]
        print(f'\n{h}: {col}'); print(p.round(4).to_string())

In [ ]:
# 8 · reproducibility check: w32 is last run's "small". Do the numbers repeat?
# Test scores of last run's "small" runs (Kaggle, Tesla T4). A different GPU can
# shift results by a few thousandths; differences under 0.01 are ties.
PREV = {('mlp', 'none'): 0.9184, ('mlp', 'hidden'): 0.9201, ('mlp', 'decay'): 0.9204,
        ('mlp', 'input'): 0.9171, ('mlp', 'input5'): 0.9160, ('mlp', 'all'): 0.9165,
        ('tabresnet', 'none'): 0.9154, ('tabresnet', 'hidden'): 0.9191, ('tabresnet', 'decay'): 0.9165,
        ('tabresnet', 'input'): 0.9210, ('tabresnet', 'input5'): 0.9183, ('tabresnet', 'all'): 0.9194}
w = sh[sh['size'] == 'w32']
out = []
for (h, r), old in PREV.items():
    v = w[(w['head'] == h) & (w['reg'] == r)]
    if len(v):
        out.append({'head': h, 'regularisation': r, 'last run': old, 'this run': float(v['test'].iloc[0]),
                    'difference': float(v['test'].iloc[0]) - old})
if out:
    d = pd.DataFrame(out)
    print(d.round(4).to_string(index=False))
    print(f'\nlargest difference {d["difference"].abs().max():.4f}:',
          'reproduces (inside the 0.01 noise)' if d['difference'].abs().max() < 0.01
          else 'LARGER than the noise; worth a look before trusting the ladder')
else:
    print('no w32 runs yet')

In [ ]:
# 9 · the charts
from IPython.display import Image, display
for f in ['fig2_capacity.png', 'fig1_heads.png', 'fig3_curves.png']:
    p = f'{ROOT}/{f}'
    if os.path.exists(p):
        print('=' * 30, f, '=' * 30); display(Image(p))

In [ ]:
# 10 · BUNDLE AND DOWNLOAD everything: logs, results, tables, charts, README
#      (re-aggregates first, so the bundle is complete even after a partial session)
!python -u run_deep_head_suite.py {SUITE} --aggregate-only --bundle 2>&1 | tail -3
z = sorted(glob.glob('results/deep_head_tiny_*.zip'), key=os.path.getmtime)[-1]
shutil.copy(z, f'{DRIVE}/{os.path.basename(z)}')
shutil.copytree(ROOT, RUNS, dirs_exist_ok=True)
print(f'\n{os.path.basename(z)}   {os.path.getsize(z)/1e6:.1f} MB')
print('a copy is also on Drive:', f'{DRIVE}/{os.path.basename(z)}')
from google.colab import files
files.download(z)